In [108]:
import pandas as pd

df = pd.read_csv("detentionstints.csv")

/var/folders/x8/ddg8bl2d01bd6hq3vchn02t00000gn/T/ipykernel_72049/1824939824.py:3: DtypeWarning: Columns (0: likely_duplicate) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("detentionstints.csv")


In [109]:
#Filter to all sints in NYC AOR 

all_NY_stints = df.query("book_in_aor == 'New York City Area of Responsibility'")

In [110]:
#Filter original df to unique identifiers belonging to those who did a detention stint in NYCAOR
NY_stints_ids = all_NY_stints['unique_identifier'].unique()

In [111]:
#Dataframe of all stints of every person who ever did a stint in NYCAOR
held_in_NY_ever = df[df['unique_identifier'].isin(NY_stints_ids)]

In [112]:
cols = ['book_in_date_time','stay_book_in_date_time', 'book_out_date_time', 'stay_book_out_date_time']
held_in_NY_ever[cols] = held_in_NY_ever[cols].apply(lambda x: pd.to_datetime(x, utc=True))

In [113]:
held_in_NY_ever = held_in_NY_ever.drop(columns=['msc_charge','msc_sentence_days','msc_sentence_months','msc_sentence_years','most_serious_conviction_code'])

In [114]:
# find the earliest stint for each person
earliest = held_in_NY_ever.loc[held_in_NY_ever.groupby('unique_identifier')['book_in_date_time'].idxmin()]

# keep only people whose earliest stint was in NY
first_in_NY_ids = earliest[earliest['book_in_aor'] == 'New York City Area of Responsibility']['unique_identifier']

# filter the full df down to just those people
NewYorkers = held_in_NY_ever[held_in_NY_ever['unique_identifier'].isin(first_in_NY_ids)]

#this is a dataframe of all individual stints where the person in question was first detained in New York.

In [115]:
#cutoff between Biden and Trump 
inauguration = pd.Timestamp('2025-01-20', tz='UTC')

# Get the earliest book_in_date_time per person
earliest_per_person = NewYorkers.groupby('unique_identifier')['book_in_date_time'].min()

# Separate the IDs into two groups
biden_ids = earliest_per_person[earliest_per_person < inauguration].index
trump_ids = earliest_per_person[earliest_per_person >= inauguration].index

# Filter the full NewYorkers df by those IDs
NYdetained_Biden = NewYorkers[NewYorkers['unique_identifier'].isin(biden_ids)]
NYdetained_Trump = NewYorkers[NewYorkers['unique_identifier'].isin(trump_ids)]


In [283]:
#Then realize you should do an equivalent time frame for Biden 
#From inauguration to data release = 414 days 
#414 days before inauguration = December 3 2023, 2023-12-03 
Biden414 = pd.Timestamp('2023-12-03', tz='UTC')

NewBiden_ids = earliest_per_person[(earliest_per_person >= Biden414) & (earliest_per_person < inauguration)].index

NYdetained_Biden2 = NewYorkers[NewYorkers['unique_identifier'].isin(NewBiden_ids)]

In [503]:
#Number of people detained under Trump

NYdetained_Trump['unique_identifier'].nunique()

9222

In [504]:
#Number of people detained under Biden

NYdetained_Biden2['unique_identifier'].nunique()

3488

In [285]:
#Add column for federal court circuits based on state 

state_to_circuit = {
    # 1st Circuit
    'ME': '1st', 'NH': '1st', 'MA': '1st', 'RI': '1st', 'PR': '1st',
    # 2nd Circuit
    'NY': '2nd', 'CT': '2nd', 'VT': '2nd',
    # 3rd Circuit
    'PA': '3rd', 'NJ': '3rd', 'DE': '3rd', 'VI': '3rd',
    # 4th Circuit
    'MD': '4th', 'VA': '4th', 'WV': '4th', 'NC': '4th', 'SC': '4th',
    # 5th Circuit
    'TX': '5th', 'LA': '5th', 'MS': '5th',
    # 6th Circuit
    'MI': '6th', 'OH': '6th', 'KY': '6th', 'TN': '6th',
    # 7th Circuit
    'IL': '7th', 'IN': '7th', 'WI': '7th',
    # 8th Circuit
    'MN': '8th', 'IA': '8th', 'MO': '8th', 'AR': '8th',
    'NE': '8th', 'ND': '8th', 'SD': '8th',
    # 9th Circuit
    'CA': '9th', 'OR': '9th', 'WA': '9th', 'AK': '9th', 'HI': '9th',
    'AZ': '9th', 'NV': '9th', 'ID': '9th', 'MT': '9th', 'GU': '9th', 'MP': '9th',
    # 10th Circuit
    'CO': '10th', 'WY': '10th', 'UT': '10th',
    'KS': '10th', 'OK': '10th', 'NM': '10th',
    # 11th Circuit
    'FL': '11th', 'GA': '11th', 'AL': '11th',
    # DC Circuit
    'DC': 'DC',
}

NYdetained_Trump['circuit'] = NYdetained_Trump['state'].map(state_to_circuit)
NYdetained_Biden2['circuit'] = NYdetained_Biden2['state'].map(state_to_circuit)


In [501]:
#Finding all those detained under Trump who are still in detention
#1. filter for all those who have no stay book out time (stay = entire time in detention system, made up of all of a person's stints)

stilldetainedTrump = NYdetained_Trump[NYdetained_Trump['stay_book_out_date_time'].isna()]


In [240]:
#checking 
stilldetainedTrump['stay_release_reason'].value_counts(dropna=False)

stay_release_reason
NaN    6323
Name: count, dtype: int64

In [120]:
stilldetainedTrump['case_category'].value_counts(dropna=False)

case_category
[8B] Excludable / Inadmissible - Under Adjudication by IJ                 2603
[8C] Excludable / Inadmissible - Administrative Final Order Issued        1290
[8D] Excludable / Inadmissible - Under Adjudication by BIA                 595
NaN                                                                        410
[2A] Deportable - Under Adjudication by IJ                                 393
[3] Deportable - Administratively Final Order                              339
[16] Reinstated Final Order                                                221
[2B] Deportable - Under Adjudication by BIA                                135
[8A] Excludable / Inadmissible - Hearing Not Commenced                     128
[1A] Voluntary Departure - Un-Expired and Un-Extended Departure Period      94
[8F] Expedited Removal                                                      43
[10] Visa Waiver Deportation / Removal                                      34
[5C] Relief Granted - Withholding of D

In [121]:
#Total number of NYers detained under Trump still in detention
stilldetainedTrump['unique_identifier'].nunique()

1687

In [122]:
#Find the latest stint per person as of data's release
latest_stilldetainedTrump = stilldetainedTrump.loc[stilldetainedTrump.groupby('unique_identifier')['book_in_date_time'].idxmax()]

In [241]:
#Find the circuit of these stints 
latest_stilldetainedTrump['circuit'].value_counts()

circuit
5th     622
2nd     364
3rd     236
9th     179
10th    108
7th      51
11th     42
6th      41
4th      15
8th       1
1st       1
Name: count, dtype: int64

In [252]:
latest_stilldetainedTrump['state'].value_counts()

state
NY    364
LA    302
TX    206
NJ    130
CA    122
MS    114
PA    106
NM     90
IN     51
AZ     47
MI     35
GA     27
FL     15
VA     15
CO     13
WA     10
KY      6
KS      4
OK      1
IA      1
RI      1
Name: count, dtype: int64

In [251]:
latest_stilldetainedTrump['book_in_aor'].value_counts()

book_in_aor
New Orleans Area of Responsibility      416
New York City Area of Responsibility    352
El Paso Area of Responsibility          134
Philadelphia Area of Responsibility     133
Newark Area of Responsibility           130
San Francisco Area of Responsibility     71
Houston Area of Responsibility           68
Chicago Area of Responsibility           61
San Diego Area of Responsibility         52
San Antonio Area of Responsibility       52
Phoenix Area of Responsibility           44
Harlingen Area of Responsibility         42
Detroit Area of Responsibility           35
Atlanta Area of Responsibility           27
Miami Area of Responsibility             15
Washington Area of Responsibility        15
Denver Area of Responsibility            13
Buffalo Area of Responsibility           12
Seattle Area of Responsibility           10
Los Angeles Area of Responsibility        2
Dallas Area of Responsibility             1
St. Paul Area of Responsibility           1
Boston Area of Respo

In [137]:
#Number deported under Biden 
#(There are a bunch of different columns that one could use but I've found stay release reason to make the most sense 
#and the DDP people said using it for this purpose is fine, when I asked them) 
Bidenremoved = NYdetained_Biden2.query("stay_release_reason == 'Removed'")
Bidenremoved['unique_identifier'].nunique()

737

In [138]:
#percent of detainees removed under Biden

(Bidenremoved['unique_identifier'].nunique()/NYdetained_Biden2['unique_identifier'].nunique())*100

21.129587155963304

In [136]:
NYdetained_Trump['unique_identifier'].nunique()

9222

In [139]:
#Number removed under Trump 
Trumpremoved = NYdetained_Trump.query("stay_release_reason == 'Removed'")
Trumpremoved['unique_identifier'].nunique()

5899

In [140]:
#percent of detainees removed (so far) under Trump
(Trumpremoved['unique_identifier'].nunique()/NYdetained_Trump['unique_identifier'].nunique())*100

63.966601604857956

In [141]:
#Finding how many detainees with unsettled cases (not including cases on appeal) were removed or transferred outside of region

#Filter for people with ongoing cases 

Trumpcaseongoing = NYdetained_Trump[NYdetained_Trump['case_category'].isin (['[8B] Excludable / Inadmissible - Under Adjudication by IJ','[8A] Excludable / Inadmissible - Hearing Not Commenced','[2A] Deportable - Under Adjudication by IJ'])]


In [142]:
Trumpcaseongoing['unique_identifier'].nunique()

2466

In [143]:
Trumpcaseongoing.groupby("case_category", dropna=False)["unique_identifier"].nunique()

case_category
[2A] Deportable - Under Adjudication by IJ                    301
[8A] Excludable / Inadmissible - Hearing Not Commenced        137
[8B] Excludable / Inadmissible - Under Adjudication by IJ    2032
Name: unique_identifier, dtype: int64

In [144]:
Trumpcaseongoing.groupby("stay_release_reason", dropna=False)["unique_identifier"].nunique()

stay_release_reason
Bonded Out - Field Office                                          4
Bonded Out - IJ                                                  347
Court Ordered                                                      5
Order of Recognizance - Humanitarian                              65
Order of Supervision - Humanitarian                                3
Order of recognizance                                            677
Order of supervision                                               1
Paroled                                                           21
Paroled - Humanitarian                                             1
Proceedings Terminated                                            17
Relief Granted by IJ                                              47
Removed                                                          334
Transferred                                                       25
U.S. Marshals or other agency (explain in Detention Comments)     34
Voluntary depa

In [274]:
#Find the most recent stint for each person in Trumpcaseongoing, and then the location of that stint 

TrumpcaseongoingLatest = Trumpcaseongoing.loc[Trumpcaseongoing.groupby('unique_identifier')['book_in_date_time'].idxmax()]
TrumpcaseongoingLatest['state'].value_counts()

state
NY    996
TX    342
LA    269
NJ    207
CA    170
PA    110
NM     94
AZ     69
MS     63
MI     33
GA     30
IN     18
FL     18
CO     12
VA     12
WA      6
KY      6
MA      2
Name: count, dtype: int64

In [260]:
TrumpcaseongoingLatest = Trumpcaseongoing.loc[Trumpcaseongoing.groupby('unique_identifier')['book_in_date_time'].idxmax()]
TrumpcaseongoingLatest['circuit'].value_counts()

circuit
2nd     996
5th     674
3rd     317
9th     245
10th    106
11th     48
6th      39
7th      18
4th      12
1st       2
Name: count, dtype: int64

In [457]:
TrumpcaseongoingLatest['book_in_aor'].value_counts()

book_in_aor
New York City Area of Responsibility    978
New Orleans Area of Responsibility      332
Newark Area of Responsibility           203
El Paso Area of Responsibility          180
Philadelphia Area of Responsibility     119
Harlingen Area of Responsibility        104
San Francisco Area of Responsibility     93
Houston Area of Responsibility           82
San Diego Area of Responsibility         80
San Antonio Area of Responsibility       69
Phoenix Area of Responsibility           65
Detroit Area of Responsibility           33
Atlanta Area of Responsibility           30
Chicago Area of Responsibility           24
Buffalo Area of Responsibility           22
Miami Area of Responsibility             18
Denver Area of Responsibility            12
Washington Area of Responsibility        12
Seattle Area of Responsibility            6
Boston Area of Responsibility             2
Dallas Area of Responsibility             1
Los Angeles Area of Responsibility        1
Name: count, dtype: 

In [531]:
TrumpcaseongoingLatest[TrumpcaseongoingLatest["book_in_aor"] == 'New York City Area of Responsibility']['unique_identifier'].nunique()

978

In [538]:
#Trump detainees with ongoing cases who were transferred out of NYC AOR
TrumpcaseongoingLatest['unique_identifier'].nunique() - 978

1488

In [539]:
#Percent of NYers with ongoing immigration cases transferred out of NYC AOR 
(1488/2466)*100

60.34063260340633

In [458]:
#number of all NYers currently detained in the 5th circuit
currently5 = latest_stilldetainedTrump.query("circuit == '5th'")

IDscurrently5 = currently5['unique_identifier']

trackingcurrent5 = NYdetained_Trump[NYdetained_Trump['unique_identifier'].isin(IDscurrently5)]

firstdetained5 = trackingcurrent5.groupby('unique_identifier')['stay_book_in_date_time'].first()

stayduration5 = (datafiled - firstdetained5.dt.tz_localize(None)).dt.days

currently5['unique_identifier'].nunique()


622

In [452]:
stayduration5.median()

np.float64(104.0)

In [454]:
stayduration5.mean()

np.float64(125.39871382636656)

In [149]:
#everyone still detained with ongoing cases
Trumpcaseongoing_stilldetained = stilldetainedTrump[stilldetainedTrump['case_category'].isin (['[8B] Excludable / Inadmissible - Under Adjudication by IJ','[8A] Excludable / Inadmissible - Hearing Not Commenced','[2A] Deportable - Under Adjudication by IJ'])]
Trumpcaseongoing_stilldetained['unique_identifier'].nunique()

852

In [276]:
#Find the latest stint for them 
Trumpcaseongoing_stilldetainedLatest = Trumpcaseongoing_stilldetained.loc[Trumpcaseongoing_stilldetained.groupby('unique_identifier')['book_in_date_time'].idxmax()]

Trumpcaseongoing_stilldetainedLatest['book_in_aor'].value_counts()

book_in_aor
New York City Area of Responsibility    210
New Orleans Area of Responsibility      138
El Paso Area of Responsibility           87
Philadelphia Area of Responsibility      67
Newark Area of Responsibility            63
San Francisco Area of Responsibility     59
San Diego Area of Responsibility         37
Houston Area of Responsibility           35
San Antonio Area of Responsibility       25
Chicago Area of Responsibility           23
Detroit Area of Responsibility           22
Phoenix Area of Responsibility           20
Atlanta Area of Responsibility           18
Harlingen Area of Responsibility         12
Denver Area of Responsibility            10
Miami Area of Responsibility             10
Buffalo Area of Responsibility            8
Washington Area of Responsibility         5
Seattle Area of Responsibility            3
Name: count, dtype: int64

In [277]:
Trumpcaseongoing_stilldetainedLatest['circuit'].value_counts()

circuit
5th     234
2nd     218
3rd     122
9th     119
10th     73
11th     28
6th      27
7th      18
4th       5
Name: count, dtype: int64

In [279]:
Trumpcaseongoing_stilldetainedLatest['state'].value_counts()

state
NY    218
LA    104
TX     96
CA     94
NJ     63
NM     63
PA     59
MS     34
AZ     22
MI     22
GA     18
IN     18
CO     10
FL     10
KY      5
VA      5
WA      3
Name: count, dtype: int64

In [516]:
# Get all stints for people in each group that are out of the SECOND AND THIRD (and first but that number is very small) CIRCUIT

#They have similar politics. Legal Aid attorney told me that advocating for a client in NJ and PA (third circuit) 
#is much the same as if they're in NY

# These circuits also geographically make up the Northeast from PA to Maine

trumptransfers_no23circ = NYdetained_Trump[~ NYdetained_Trump['circuit'].isin(['2nd', '3rd', '1st'])]

bidentransfers_no23circ = NYdetained_Biden2[~ NYdetained_Biden2['circuit'].isin(['2nd', '3rd', '1st'])]

# Calculate days from stay start to out-of-region arrival
trumptransfers_no23circ['days_to_transfer'] = (
    pd.to_datetime(trumptransfers_no23circ['book_in_date_time']) - 
    pd.to_datetime(trumptransfers_no23circ['stay_book_in_date_time'])
).dt.days

bidentransfers_no23circ['days_to_transfer'] = (
    pd.to_datetime(bidentransfers_no23circ['book_in_date_time']) - 
    pd.to_datetime(bidentransfers_no23circ['stay_book_in_date_time'])
).dt.days

# Keep only the FIRST out-of-region stint per person
trumpfirst_no23circ = trumptransfers_no23circ.sort_values('book_in_date_time').groupby('unique_identifier').first()
bidenfirst_no23circ = bidentransfers_no23circ.sort_values('book_in_date_time').groupby('unique_identifier').first()

# Compare medians
print("Trump median days to transfer:", trumpfirst_no23circ['days_to_transfer'].median())
print("Biden median days to transfer:", bidenfirst_no23circ['days_to_transfer'].median())

Trump median days to transfer: 7.0
Biden median days to transfer: 35.0


In [323]:
trumpfirst_no23circ['days_to_transfer'].value_counts().sort_index().head(10)

days_to_transfer
0    199
1    521
2    612
3    612
4    476
5    479
6    458
7    454
8    420
9    306
Name: count, dtype: int64

In [324]:

Trumpcircuitunder5 = trumpfirst_no23circ[trumpfirst_no23circ['days_to_transfer'] <= 4].index.nunique()
Trumpcircuitunder5

2420

In [524]:
#Percent of NYers with ongoing immigration cases transferred out of 1st, 2nd & 3rd circuit
Trumpcaseongoing[~Trumpcaseongoing['circuit'].isin(['1st', '2nd', '3rd'])]['unique_identifier'].nunique()


1201

In [526]:
(Trumpcaseongoing[~Trumpcaseongoing['circuit'].isin(['1st', '2nd', '3rd'])]['unique_identifier'].nunique())/(Trumpcaseongoing['unique_identifier'].nunique())

0.4870235198702352

In [514]:
(bidenfirst_no23circ.index.nunique())/(NYdetained_Biden2['unique_identifier'].nunique())

0.19208715596330275

In [506]:
#number of people detained by Trump who were transferred outside the 1st 2nd 3rd circuits
#(unique id is the index)
trumpfirst_no23circ.index.nunique()

7058

In [513]:
#percent of people detained by Trump (out of total detained by him) who were transferred outside the 1st 2nd 3rd circuits
(trumpfirst_no23circ.index.nunique())/(NYdetained_Trump['unique_identifier'].nunique())


0.7653437432227282

In [328]:
print(f"Out of {NYdetained_Trump['unique_identifier'].nunique()} people detained under Trump, {trumpfirst_no23circ.index.nunique()} have experienced being transferred outside of the 1st, 2nd and 3rd circuits.")
print(f"{Trumpcircuitunder5}, or {(Trumpcircuitunder5/NYdetained_Trump['unique_identifier'].nunique())*100:.1f}% of all detainees, were transferred out in under 5 days.")


Out of 9222 people detained under Trump, 7058 have experienced being transferred outside of the 1st, 2nd and 3rd circuits.
2420, or 26.2% of all detainees, were transferred out in under 5 days.


In [186]:
#removed tends to include VD 
Trumpcaseongoingremoved['unique_identifier'].nunique()

334

In [183]:
#poking around
Trumpcaseongoingremoved.groupby('book_in_criminality', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)


book_in_criminality
3 Other Immigration Violator    264
1 Convicted Criminal             41
2 Pending Criminal Charges       29
Name: unique_identifier, dtype: int64

In [184]:
Trumpcaseongoingremoved.groupby('suspected_gang_yes_no', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)


suspected_gang_yes_no
NO     319
YES     15
Name: unique_identifier, dtype: int64

In [185]:
Trumpcaseongoingremoved.groupby('case_status', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)


case_status
3-Voluntary Departure Confirmed          264
E-Charging Document Canceled by ICE       19
A-Proceedings Terminated                  18
ACTIVE                                    13
0-Withdrawal Permitted - I-275 Issued     10
5-Title 50 Expulsion                      10
Name: unique_identifier, dtype: int64

In [187]:
Trumpcaseongoing.groupby('case_status', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)


case_status
ACTIVE                                          1997
3-Voluntary Departure Confirmed                  332
A-Proceedings Terminated                          54
B-Relief Granted                                  38
E-Charging Document Canceled by ICE               28
0-Withdrawal Permitted - I-275 Issued             10
5-Title 50 Expulsion                              10
L-Legalization - Permanent Residence Granted       1
Name: unique_identifier, dtype: int64

In [509]:
#Using Deportation Data Project methodology for accounting for VDS 

vd_case_categories = [
    '[1A] Voluntary Departure - Un-Expired and Un-Extended Departure Period',
    '[1B] Voluntary Departure - Extended Departure Period',
    '[5E] Relief Granted - Extended Voluntary Departure'
]

vd_case_statuses = [
    '3-Voluntary Departure Confirmed',
    '9-VR Witnessed',
    '0-Withdrawal Permitted - I-275 Issued'
]

vd_release_reasons = [
    'Voluntary departure',
    'Voluntary Return'
]

NYdetained_Trump['VD?'] = (
    NYdetained_Trump['case_category'].isin(vd_case_categories) |
    NYdetained_Trump['case_status'].isin(vd_case_statuses) |
    NYdetained_Trump['stay_release_reason'].isin(vd_release_reasons)
)

In [358]:
Trumpcaseongoing['VD?'] = (
    Trumpcaseongoing['case_category'].isin(vd_case_categories) |
    Trumpcaseongoing['case_status'].isin(vd_case_statuses) |
    Trumpcaseongoing['stay_release_reason'].isin(vd_release_reasons)
)


Trumpcaseongoing.groupby('VD?', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

VD?
False    2122
True      344
Name: unique_identifier, dtype: int64

In [353]:
#people with ongoing cases who have chosen Voluntary Departure
TrumpcaseongoingVD = Trumpcaseongoing[Trumpcaseongoing['VD?'] == True]
TrumpcaseongoingVD['unique_identifier'].nunique()

344

In [359]:
#checking how many people marked "removed" were VD
Trumpcaseongoingremoved['VD?'] = (
    Trumpcaseongoingremoved['case_category'].isin(vd_case_categories) |
    Trumpcaseongoingremoved['case_status'].isin(vd_case_statuses) |
    Trumpcaseongoingremoved['stay_release_reason'].isin(vd_release_reasons)
)


Trumpcaseongoingremoved.groupby('VD?', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

VD?
True     274
False     60
Name: unique_identifier, dtype: int64

In [543]:
Allbidenlaststint = NYdetained_Biden2.loc[NYdetained_Biden2.groupby('unique_identifier')['book_in_date_time'].idxmax()]
Allbidenlaststint['book_in_aor'].value_counts()

book_in_aor
New York City Area of Responsibility    2632
New Orleans Area of Responsibility       479
Philadelphia Area of Responsibility      140
Newark Area of Responsibility             52
Harlingen Area of Responsibility          48
Miami Area of Responsibility              26
Phoenix Area of Responsibility            20
Dallas Area of Responsibility             19
El Paso Area of Responsibility            17
Buffalo Area of Responsibility            15
Houston Area of Responsibility            11
Chicago Area of Responsibility            10
San Antonio Area of Responsibility         9
Atlanta Area of Responsibility             3
Denver Area of Responsibility              3
Boston Area of Responsibility              2
Washington Area of Responsibility          1
Seattle Area of Responsibility             1
Name: count, dtype: int64

In [544]:
# Number transferred out of NYCAOR under Biden 
BidentransferredoutAOR = Allbidenlaststint[Allbidenlaststint['book_in_aor'] != 'New York City Area of Responsibility']
BidentransferredoutAOR['unique_identifier'].nunique()

856

In [545]:
Alltrumplaststint = NYdetained_Trump.loc[NYdetained_Trump.groupby('unique_identifier')['book_in_date_time'].idxmax()]
Alltrumplaststint['book_in_aor'].value_counts()

book_in_aor
New Orleans Area of Responsibility      3259
New York City Area of Responsibility    1602
Harlingen Area of Responsibility        1073
El Paso Area of Responsibility           613
Houston Area of Responsibility           553
Phoenix Area of Responsibility           409
Newark Area of Responsibility            405
San Antonio Area of Responsibility       333
Philadelphia Area of Responsibility      290
San Diego Area of Responsibility         160
San Francisco Area of Responsibility     122
Miami Area of Responsibility              68
Chicago Area of Responsibility            65
Atlanta Area of Responsibility            58
Detroit Area of Responsibility            58
Washington Area of Responsibility         54
Buffalo Area of Responsibility            33
Denver Area of Responsibility             18
Dallas Area of Responsibility             17
Seattle Area of Responsibility            15
Los Angeles Area of Responsibility         9
Boston Area of Responsibility              

In [548]:
# Number transferred out of NYCAOR under Trump 
TrumptransferredoutAOR = Alltrumplaststint[Alltrumplaststint['book_in_aor'] != 'New York City Area of Responsibility']
TrumptransferredoutAOR['unique_identifier'].nunique()

7620

In [498]:


#THE REST OF THIS NOTEBOOK WAS NOT ULTIMATELY USED!

#I was just poking around in the data a lot. Everything I actually used and which is relevant should be included above this cell
#I just didn't want to scrap the other things I was working on






#Finding average stay duration of those in New Orleans field office area
#1. make a df of all stints for people who are currently in NOLA 

currentlyNewOrleans = latest_stilldetainedTrump.query("book_in_aor == 'New Orleans Area of Responsibility'")

IDscurrentlyNewOrleans = currentlyNewOrleans['unique_identifier']

trackingcurrentNOLA = NYdetained_Trump[NYdetained_Trump['unique_identifier'].isin(IDscurrentlyNewOrleans)]

#check
trackingcurrentNOLA['unique_identifier'].nunique()

416

In [125]:
#2. Find the stay duration for each person 
NOLAfirstdetained = trackingcurrentNOLA.groupby('unique_identifier')['stay_book_in_date_time'].min()

datafiled = pd.Timestamp('2026-03-10')

NOLAstayduration = (datafiled - NOLAfirstdetained.dt.tz_localize(None)).dt.days

NOLAstayduration.median()

np.float64(93.0)

In [253]:
#Trying same for El Paso 
currentlyElPaso = latest_stilldetainedTrump.query("book_in_aor == 'El Paso Area of Responsibility'")

IDscurrentlyElPaso = currentlyElPaso['unique_identifier']

trackingcurrentElPaso = NYdetained_Trump[NYdetained_Trump['unique_identifier'].isin(IDscurrentlyElPaso)]

ElPasofirstdetained = trackingcurrentElPaso.groupby('unique_identifier')['stay_book_in_date_time'].first()

ElPasostayduration = (datafiled - ElPasofirstdetained.dt.tz_localize(None)).dt.days

ElPasostayduration.median()

np.float64(112.0)

In [127]:
currentlyElPaso['detention_facility'].value_counts()

detention_facility
OTERO CO PROCESSING CENTER           59
ERO EL PASO CAMP EAST MONTANA        36
CIBOLA COUNTY CORRECTIONAL CENTER    18
TORRANCE/ESTANCIA, NM                13
EL PASO SPC                           8
Name: count, dtype: int64

In [128]:
# https://elpasomatters.org/2026/03/12/ice-replacing-camp-east-montana-detention-center-operator-contract-washington-post/ 
# there have been measles at ERO EL PASO CAMP EAST MONTANA where 36 New Yorkers were still detained as of March 10

currentEroElPaso = currentlyElPaso.query("detention_facility == 'ERO EL PASO CAMP EAST MONTANA'")

In [129]:
#Find month each of these people were placed here, for researching how many experienced what wave of outbreak
currentEroElPaso['book_in_date_time'].dt.to_period('M').value_counts().sort_index()

/var/folders/x8/ddg8bl2d01bd6hq3vchn02t00000gn/T/ipykernel_72049/509717478.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  currentEroElPaso['book_in_date_time'].dt.to_period('M').value_counts().sort_index()


book_in_date_time
2025-08     3
2025-09    10
2025-11     4
2025-12     6
2026-01     4
2026-02     9
Freq: M, Name: count, dtype: int64

In [130]:
#check
currentEroElPaso['unique_identifier'].nunique()

36

In [131]:
#severe medical neglect in North Lake, people are hunger striking
#https://www.lemkininstitute.com/items-5/trump-administration%E2%80%99s-mass-deportation-operations%3A-update-april-2026
#https://www.latintimes.com/lawmakers-activists-demand-answers-after-ice-detainees-launch-hunger-strike-michigan-facility-596931
latest_stilldetainedTrump.query("book_in_aor == 'Detroit Area of Responsibility'")['detention_facility'].value_counts()

detention_facility
NORTH LAKE CORRECTIONAL FACILITY     24
CALHOMI CALHOUN CO., BATTLE CR,MI    10
MONROE COUNTY DETENTION-DORM          1
Name: count, dtype: int64

In [496]:


Trumpcaseongoingremoved.groupby('case_status', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

case_status
3-Voluntary Departure Confirmed          264
E-Charging Document Canceled by ICE       19
A-Proceedings Terminated                  18
ACTIVE                                    13
0-Withdrawal Permitted - I-275 Issued     10
5-Title 50 Expulsion                      10
Name: unique_identifier, dtype: int64

In [200]:
Trumpcaseongoingremoved_noVD = Trumpcaseongoingremoved[Trumpcaseongoingremoved['VD?'] == False]
Trumpcaseongoingremoved_noVD.groupby('case_status', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

case_status
E-Charging Document Canceled by ICE    19
A-Proceedings Terminated               18
ACTIVE                                 13
5-Title 50 Expulsion                   10
Name: unique_identifier, dtype: int64

In [201]:
Trumpcaseongoingremoved_noVD.groupby('stay_release_reason', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

stay_release_reason
Removed    60
Name: unique_identifier, dtype: int64

In [208]:
Trumpcaseongoing.groupby('stay_release_reason', dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

stay_release_reason
NaN                                                              852
Order of recognizance                                            677
Bonded Out - IJ                                                  347
Removed                                                          334
Voluntary departure                                               67
Order of Recognizance - Humanitarian                              65
Relief Granted by IJ                                              47
U.S. Marshals or other agency (explain in Detention Comments)     34
Transferred                                                       25
Paroled                                                           21
Proceedings Terminated                                            17
Court Ordered                                                      5
Bonded Out - Field Office                                          4
Order of Supervision - Humanitarian                                3
Order of super

In [352]:
#these next columns were just to suss out how reliable case_category was and keep exploring the data
Finalorderyes = NYdetained_Trump.query("final_order_yes_no == 'YES'")
Finalorderyes['case_category'].value_counts(dropna=False)

case_category
[8C] Excludable / Inadmissible - Administrative Final Order Issued     13697
[16] Reinstated Final Order                                             3308
[3] Deportable - Administratively Final Order                           1558
NaN                                                                     1302
[8F] Expedited Removal                                                   650
[10] Visa Waiver Deportation / Removal                                   207
[12] Judicial Deportation / Removal                                       81
[11] Administrative Deportation / Removal                                 70
[5D] Final Order of Deportation / Removal - Deferred Action Granted       19
[8H] Expedited Removal - Status Claim Referral                            15
[5C] Relief Granted - Withholding of Deportation / Removal                11
[5F] Unable to Obtain Travel Document                                      5
[8E] Inadmissible - ICE Fugitive                              

In [218]:
#Not a single person with "final order = yes" had any of the "case ongoing" case categories.

In [219]:
Finalorderyes['unique_identifier'].nunique()

4934

In [220]:
Finalorderno = NYdetained_Trump.query("final_order_yes_no == 'NO'")
Finalorderno['case_category'].value_counts(dropna=False)

case_category
[8B] Excludable / Inadmissible - Under Adjudication by IJ                 6589
[8C] Excludable / Inadmissible - Administrative Final Order Issued        3608
[1A] Voluntary Departure - Un-Expired and Un-Extended Departure Period    2054
[2A] Deportable - Under Adjudication by IJ                                1099
[8D] Excludable / Inadmissible - Under Adjudication by BIA                 728
NaN                                                                        651
[8A] Excludable / Inadmissible - Hearing Not Commenced                     371
[3] Deportable - Administratively Final Order                              221
[2B] Deportable - Under Adjudication by BIA                                161
[16] Reinstated Final Order                                                 82
[8F] Expedited Removal                                                      53
[1B] Voluntary Departure - Extended Departure Period                        34
[5E] Relief Granted - Extended Volunta

In [221]:
Trumpcaseongoing['final_order_yes_no'].value_counts(dropna=False)

final_order_yes_no
NO    8059
Name: count, dtype: int64

In [222]:
Finalorderyes.groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique()

stay_release_reason
Bonded Out - Field Office                                           1
Bonded Out - IJ                                                    11
Died                                                                1
Order of Recognizance - Humanitarian                                1
Order of Supervision - Humanitarian                                 4
Order of Supervision - No SLRRFF                                    1
Order of Supervision - Re-Release                                   4
Order of recognizance                                               9
Order of supervision                                               60
Paroled                                                             5
Proceedings Terminated                                              2
Relief Granted by IJ                                                2
Removed                                                          4383
Transferred                                                        60


In [223]:
Finalorderyes.groupby('case_status',dropna=False)['unique_identifier'].nunique()

case_status
0-Withdrawal Permitted - I-275 Issued       7
3-Voluntary Departure Confirmed           191
6-Deported/Removed - Deportability        579
7-Died                                      1
8-Excluded/Deported/Removed               189
8-Excluded/Removed - Inadmissibility     3427
ACTIVE                                    561
B-Relief Granted                            2
Name: unique_identifier, dtype: int64

In [224]:
Finalorderyes[Finalorderyes['stay_release_reason'].isna()].groupby('case_status',dropna=False)['unique_identifier'].nunique()

case_status
6-Deported/Removed - Deportability      1
ACTIVE                                419
Name: unique_identifier, dtype: int64

In [225]:
#stay release reason being NaN correlate almost perfectly to case status active here

In [226]:
Finalorderno[Finalorderno['case_category'] == '[3] Deportable - Administratively Final Order']['unique_identifier'].nunique()

52

In [227]:
Finalorderno[Finalorderno['case_category'] == '[3] Deportable - Administratively Final Order'].groupby('case_status',dropna=False)['unique_identifier'].nunique()

case_status
3-Voluntary Departure Confirmed    38
ACTIVE                             13
B-Relief Granted                    1
Name: unique_identifier, dtype: int64

In [228]:
#people with no final order but who were CC='[3] Deportable - Administratively Final Order' were mostly VD

In [229]:
Finalorderno[Finalorderno['case_category'] == '[1A] Voluntary Departure - Un-Expired and Un-Extended Departure Period'].groupby('case_status',dropna=False)['unique_identifier'].nunique()

case_status
3-Voluntary Departure Confirmed        438
ACTIVE                                  40
E-Charging Document Canceled by ICE      2
Name: unique_identifier, dtype: int64

In [230]:
Finalorderno[Finalorderno['case_category'] == '[1A] Voluntary Departure - Un-Expired and Un-Extended Departure Period'].groupby('final_charge',dropna=False)['unique_identifier'].nunique()

final_charge
ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)                                     400
IMMIGRANT WITHOUT AN IMMIGRANT VISA                                                     15
NONIMMIGRANT OVERSTAY                                                                   21
NONIMMIGRANT STUDENT OUT OF STATUS: FAILURE TO ATTEND                                    1
PREVIOUSLY ORDERED REMOVED AND ENTERED OR ATTEMPTED TO ENTER WITHOUT BEING ADMITTED      1
NaN                                                                                     42
Name: unique_identifier, dtype: int64

In [231]:
Trumpcaseongoing.groupby('final_charge',dropna=False)['unique_identifier'].nunique()

final_charge
ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)                                                            295
CONVICTION OF ONE CRIME INVOLVING MORAL TURPITUDE                                                               1
CONVICTION OR COMMISSION OF A CRIME INVOLVING MORAL TURPITUDE                                                   3
IMMIGRANT WITHOUT AN IMMIGRANT VISA                                                                            29
NONIMMIGRANT FAILURE TO MAINTAIN STATUS AFTER STATUS CHANGED                                                    2
NONIMMIGRANT OVERSTAY                                                                                          21
NONIMMIGRANT STATUS VIOLATORS: FAILED TO MAINTAIN THE NONIMMIGRANT STATUS IN WHICH THE ALIEN WAS ADMITTED       1
NaN                                                                                                          2114
Name: unique_identifier, dtype: int64

In [364]:
#vast majority of caseongoing have no final charge 
#all have no final order

In [354]:
NYdetained_Trump[NYdetained_Trump['final_charge'] == 'ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)']['unique_identifier'].nunique()

4273

In [355]:
Trumpcaseongoing.groupby('circuit')['unique_identifier'].nunique()

circuit
10th     170
11th      60
1st        3
2nd     2461
3rd     1342
4th       17
5th      888
6th       44
7th       21
8th        1
9th      307
Name: unique_identifier, dtype: int64

In [362]:

NYdetained_Trump[NYdetained_Trump['final_charge'] == 'ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)'].groupby('VD?')['unique_identifier'].nunique()


VD?
False    2691
True     1582
Name: unique_identifier, dtype: int64

In [363]:
NYdetained_Trump[NYdetained_Trump['final_charge'] == 'ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)'].groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique()


stay_release_reason
Bonded Out - IJ                                                     1
Order of Supervision - Humanitarian                                 1
Order of recognizance                                               3
Order of supervision                                               13
Removed                                                          4114
Transferred                                                        42
U.S. Marshals or other agency (explain in Detention Comments)      11
Voluntary Return                                                    2
Voluntary departure                                               143
NaN                                                                 2
Name: unique_identifier, dtype: int64

In [365]:
#Having the Alien Present final charge basically guarantees removal somehow. 
#whether by VD or deportation


In [366]:

NYdetained_Trump[NYdetained_Trump['final_charge'] == 'ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)'].groupby('case_category',dropna=False)['unique_identifier'].nunique()


case_category
[11] Administrative Deportation / Removal                                    4
[12] Judicial Deportation / Removal                                          4
[16] Reinstated Final Order                                                210
[1A] Voluntary Departure - Un-Expired and Un-Extended Departure Period     400
[1B] Voluntary Departure - Extended Departure Period                         4
[2A] Deportable - Under Adjudication by IJ                                   3
[2B] Deportable - Under Adjudication by BIA                                  1
[3] Deportable - Administratively Final Order                               38
[8A] Excludable / Inadmissible - Hearing Not Commenced                      17
[8B] Excludable / Inadmissible - Under Adjudication by IJ                  275
[8C] Excludable / Inadmissible - Administrative Final Order Issued        3213
[8F] Expedited Removal                                                      33
[9] VR Under Safeguards               

In [367]:
#A small but not totally insignificant number 
#of ppl w/ final charge = Alien Present, have CC = [8B] Excludable / Inadmissible - Under Adjudication by IJ

In [368]:
Trumpcaseongoing[
    (Trumpcaseongoing['case_category'] == '[8B] Excludable / Inadmissible - Under Adjudication by IJ') &
    (Trumpcaseongoing['case_status'] == 'ACTIVE')
    ].groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique()

stay_release_reason
Bonded Out - Field Office                                          2
Bonded Out - IJ                                                  202
Court Ordered                                                      4
Order of Recognizance - Humanitarian                              61
Order of Supervision - Humanitarian                                3
Order of recognizance                                            612
Paroled                                                           17
Paroled - Humanitarian                                             1
Proceedings Terminated                                             3
Relief Granted by IJ                                               9
Removed                                                           10
Transferred                                                       14
U.S. Marshals or other agency (explain in Detention Comments)     25
NaN                                                              704
Name: unique_i

In [369]:
Trumpcaseongoing[Trumpcaseongoing['case_status'] == 'ACTIVE'].groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique()

stay_release_reason
Bonded Out - Field Office                                          4
Bonded Out - IJ                                                  331
Court Ordered                                                      5
Order of Recognizance - Humanitarian                              64
Order of Supervision - Humanitarian                                3
Order of recognizance                                            670
Paroled                                                           21
Paroled - Humanitarian                                             1
Proceedings Terminated                                             3
Relief Granted by IJ                                               9
Removed                                                           13
Transferred                                                       17
U.S. Marshals or other agency (explain in Detention Comments)     33
Voluntary departure                                                1
NaN           

In [370]:
Trumpcaseongoing.groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique()

stay_release_reason
Bonded Out - Field Office                                          4
Bonded Out - IJ                                                  347
Court Ordered                                                      5
Order of Recognizance - Humanitarian                              65
Order of Supervision - Humanitarian                                3
Order of recognizance                                            677
Order of supervision                                               1
Paroled                                                           21
Paroled - Humanitarian                                             1
Proceedings Terminated                                            17
Relief Granted by IJ                                              47
Removed                                                          334
Transferred                                                       25
U.S. Marshals or other agency (explain in Detention Comments)     34
Voluntary depa

In [371]:
Trumpcaseongoing[Trumpcaseongoing['stay_release_reason'].isna()].groupby('final_charge',dropna=False)['unique_identifier'].nunique()

final_charge
ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)      1
NaN                                                   851
Name: unique_identifier, dtype: int64

In [372]:
#caseongoing with no final charge, also have no stay release reason. 
#caseongoing who do have a final charge, what is 

Trumpcaseongoing[~Trumpcaseongoing['final_charge'].isna()].groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique()

stay_release_reason
Bonded Out - IJ            1
Order of recognizance      1
Removed                  284
Transferred                7
Voluntary departure       65
NaN                        1
Name: unique_identifier, dtype: int64

In [373]:
Trumpcaseongoing[~Trumpcaseongoing['final_charge'].isna()].groupby('VD?',dropna=False)['unique_identifier'].nunique()

VD?
False     10
True     342
Name: unique_identifier, dtype: int64

In [374]:
Trumpcaseongoing[~Trumpcaseongoing['final_charge'].isna()]['unique_identifier'].nunique()

352

In [375]:
#those with ongoing cases who DO have a final charge are mostly Voluntary Departures.

In [376]:
Trumpcaseongoing[Trumpcaseongoing['VD?'] == True].groupby('final_charge',dropna=False)['unique_identifier'].nunique()

final_charge
ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)                                                           287
CONVICTION OF ONE CRIME INVOLVING MORAL TURPITUDE                                                              1
CONVICTION OR COMMISSION OF A CRIME INVOLVING MORAL TURPITUDE                                                  3
IMMIGRANT WITHOUT AN IMMIGRANT VISA                                                                           27
NONIMMIGRANT FAILURE TO MAINTAIN STATUS AFTER STATUS CHANGED                                                   2
NONIMMIGRANT OVERSTAY                                                                                         21
NONIMMIGRANT STATUS VIOLATORS: FAILED TO MAINTAIN THE NONIMMIGRANT STATUS IN WHICH THE ALIEN WAS ADMITTED      1
NaN                                                                                                            2
Name: unique_identifier, dtype: int64

In [382]:

#of the 344 caseongoings who chose VD, most had final charge of "ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)"


In [380]:
TrumpcaseongoingVD.groupby('stay_release_reason')['unique_identifier'].nunique()

stay_release_reason
Bonded Out - IJ            1
Order of recognizance      1
Removed                  274
Transferred                7
Voluntary departure       67
Name: unique_identifier, dtype: int64

In [383]:
Trumpcaseongoing[Trumpcaseongoing['VD?'] == True].groupby('final_order_yes_no',dropna=False)['unique_identifier'].nunique()

final_order_yes_no
NO    344
Name: unique_identifier, dtype: int64

In [384]:
Trumpcaseongoing[Trumpcaseongoing['VD?'] == False].groupby('final_charge',dropna=False)['unique_identifier'].nunique()

final_charge
ALIEN PRESENT WITHOUT ADMISSION OR PAROLE - (PWAs)       8
IMMIGRANT WITHOUT AN IMMIGRANT VISA                      2
NaN                                                   2112
Name: unique_identifier, dtype: int64

In [385]:
#VAST majority of caseongoing without a final charge have NOT had to do VD.
Trumpcaseongoing[Trumpcaseongoing['VD?'] == False].groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique()

stay_release_reason
Bonded Out - Field Office                                          4
Bonded Out - IJ                                                  346
Court Ordered                                                      5
Order of Recognizance - Humanitarian                              65
Order of Supervision - Humanitarian                                3
Order of recognizance                                            676
Order of supervision                                               1
Paroled                                                           21
Paroled - Humanitarian                                             1
Proceedings Terminated                                            17
Relief Granted by IJ                                              47
Removed                                                           60
Transferred                                                       18
U.S. Marshals or other agency (explain in Detention Comments)     34
NaN           

In [386]:
Trumpcaseongoing[Trumpcaseongoing['VD?'] == True].groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique()

stay_release_reason
Bonded Out - IJ            1
Order of recognizance      1
Removed                  274
Transferred                7
Voluntary departure       67
NaN                        1
Name: unique_identifier, dtype: int64

In [387]:
NYdetained_Trump[
    (NYdetained_Trump['VD?'] == False) &
    (NYdetained_Trump['stay_release_reason'] == 'Removed')
    ].groupby('final_order_yes_no',dropna=False)['unique_identifier'].nunique()


final_order_yes_no
NO       83
YES    4195
NaN       6
Name: unique_identifier, dtype: int64

In [391]:
#If this spiral was for anything, i think i can truly say now that anyone marked with ongoing cases has not received a final order by a judge. 
#Did everyone non-VD deported receive a final order 
#For the most part yes 

In [389]:
NYdetained_Trump.groupby('case_category',dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

case_category
[8C] Excludable / Inadmissible - Administrative Final Order Issued        4104
[8B] Excludable / Inadmissible - Under Adjudication by IJ                 2032
[16] Reinstated Final Order                                                841
[1A] Voluntary Departure - Un-Expired and Un-Extended Departure Period     480
NaN                                                                        479
[3] Deportable - Administratively Final Order                              398
[2A] Deportable - Under Adjudication by IJ                                 301
[8D] Excludable / Inadmissible - Under Adjudication by BIA                 188
[8F] Expedited Removal                                                     181
[8A] Excludable / Inadmissible - Hearing Not Commenced                     137
[10] Visa Waiver Deportation / Removal                                      60
[2B] Deportable - Under Adjudication by BIA                                 42
[12] Judicial Deportation / Removal   

In [392]:
NYdetained_Trump.groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

stay_release_reason
Removed                                                          5899
NaN                                                              1687
Order of recognizance                                             729
Bonded Out - IJ                                                   387
Voluntary departure                                               184
Transferred                                                       102
U.S. Marshals or other agency (explain in Detention Comments)     100
Relief Granted by IJ                                               73
Order of Recognizance - Humanitarian                               67
Order of supervision                                               62
Paroled                                                            28
Proceedings Terminated                                             19
Order of Supervision - Humanitarian                                 8
Bonded Out - Field Office                                           6


In [393]:
NYdetained_Trump[NYdetained_Trump['case_category'] == '[8B] Excludable / Inadmissible - Under Adjudication by IJ'].groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)

stay_release_reason
NaN                                                              705
Order of recognizance                                            617
Removed                                                          281
Bonded Out - IJ                                                  210
Order of Recognizance - Humanitarian                              61
Voluntary departure                                               55
Relief Granted by IJ                                              44
U.S. Marshals or other agency (explain in Detention Comments)     26
Transferred                                                       21
Paroled                                                           17
Proceedings Terminated                                            12
Court Ordered                                                      4
Order of Supervision - Humanitarian                                3
Bonded Out - Field Office                                          2
Paroled - Huma

In [394]:

TrumpcaseongoingLatest.groupby('stay_release_reason',dropna=False)['unique_identifier'].nunique().sort_values(ascending=False)


stay_release_reason
NaN                                                              852
Order of recognizance                                            676
Bonded Out - IJ                                                  342
Removed                                                          334
Voluntary departure                                               67
Order of Recognizance - Humanitarian                              65
Relief Granted by IJ                                              47
U.S. Marshals or other agency (explain in Detention Comments)     25
Paroled                                                           20
Proceedings Terminated                                            17
Transferred                                                        7
Court Ordered                                                      5
Bonded Out - Field Office                                          4
Order of Supervision - Humanitarian                                3
Order of super

In [406]:
TrumpcaseongoingLatestno23circ = TrumpcaseongoingLatest[~TrumpcaseongoingLatest['circuit'].isin(['2nd', '3rd'])]

In [410]:
TrumpcaseongoingLatestno23circ[TrumpcaseongoingLatestno23circ['stay_release_reason'] == 'Bonded Out - IJ']['circuit'].value_counts()

circuit
5th     127
9th      43
10th     16
11th     12
6th       7
4th       5
1st       1
Name: count, dtype: int64

In [414]:
#People with ongoing cases, transferred out of the second & third district, who are still in detention.

TrumpcaseongoingLatestno23circ['circuit'].value_counts()

circuit
5th     674
9th     245
10th    106
11th     48
6th      39
7th      18
4th      12
1st       2
Name: count, dtype: int64

In [416]:
TrumpcaseongoingLatestno23circ[TrumpcaseongoingLatestno23circ['stay_release_reason'].isna()].value_counts(dropna=False).sum()

np.int64(512)

In [418]:
latest_stilldetainedTrump[latest_stilldetainedTrump['stay_release_reason'].isna()]['unique_identifier'].nunique()

1687

In [423]:
TrumpcaseongoingLatestno23circ.groupby('VD?')['unique_identifier'].nunique()

VD?
False    882
True     271
Name: unique_identifier, dtype: int64

In [426]:

Trumpcaseongoing[~Trumpcaseongoing['unique_identifier'].isin(TrumpcaseongoingLatestno23circ['unique_identifier'])].groupby('VD?')['unique_identifier'].nunique()


VD?
False    1240
True       73
Name: unique_identifier, dtype: int64

In [460]:
NYdetained_Biden2.groupby('book_in_criminality')['unique_identifier'].nunique()

book_in_criminality
1 Convicted Criminal             597
2 Pending Criminal Charges       279
3 Other Immigration Violator    2641
Name: unique_identifier, dtype: int64

In [461]:
NYdetained_Trump.groupby('book_in_criminality',dropna=False)['unique_identifier'].nunique()

book_in_criminality
1 Convicted Criminal            2072
2 Pending Criminal Charges      1373
3 Other Immigration Violator    5791
Name: unique_identifier, dtype: int64

In [465]:
df[pd.to_datetime(df['stay_book_in_date_time']) < inauguration].groupby('book_in_criminality')['unique_identifier'].nunique()

book_in_criminality
1 Convicted Criminal            147822
2 Pending Criminal Charges       56181
3 Other Immigration Violator    420739
Name: unique_identifier, dtype: int64

In [468]:
fifthcircstilldetainedlateststint = latest_stilldetainedTrump[latest_stilldetainedTrump['state'].isin(['LA', 'TX', 'MS'])]
fifthcircstilldetainedlateststint['case_category'].value_counts()

case_category
[8B] Excludable / Inadmissible - Under Adjudication by IJ                 196
[8C] Excludable / Inadmissible - Administrative Final Order Issued        175
[8D] Excludable / Inadmissible - Under Adjudication by BIA                 78
[3] Deportable - Administratively Final Order                              34
[2A] Deportable - Under Adjudication by IJ                                 29
[16] Reinstated Final Order                                                23
[2B] Deportable - Under Adjudication by BIA                                15
[8A] Excludable / Inadmissible - Hearing Not Commenced                      8
[10] Visa Waiver Deportation / Removal                                      6
[1A] Voluntary Departure - Un-Expired and Un-Extended Departure Period      6
[8F] Expedited Removal                                                      5
[8H] Expedited Removal - Status Claim Referral                              2
[5C] Relief Granted - Withholding of Deportation /

In [472]:
fifthcircstilldetainedlateststint['case_status'].value_counts(dropna=False)

case_status
ACTIVE                                621
6-Deported/Removed - Deportability      1
Name: count, dtype: int64

In [471]:
fifthcircstilldetainedlateststint.info()

<class 'pandas.DataFrame'>
Index: 622 entries, 8172 to 2616031
Data columns (total 57 columns):
 #   Column                           Non-Null Count  Dtype              
---  ------                           --------------  -----              
 0   stay_book_in_date_time           622 non-null    datetime64[us, UTC]
 1   book_in_date_time                622 non-null    datetime64[us, UTC]
 2   detention_facility               622 non-null    str                
 3   book_out_date_time               0 non-null      datetime64[us, UTC]
 4   stay_book_out_date_time          0 non-null      datetime64[us, UTC]
 5   detention_release_reason         0 non-null      str                
 6   stay_book_out_date               0 non-null      str                
 7   stay_release_reason              0 non-null      str                
 8   religion                         14 non-null     str                
 9   gender                           622 non-null    str                
 10  marital_sta

In [474]:
NYdetained_Trump['VD?'] = (
    NYdetained_Trump['case_category'].isin(vd_case_categories) |
    NYdetained_Trump['case_status'].isin(vd_case_statuses) |
    NYdetained_Trump['stay_release_reason'].isin(vd_release_reasons)
)
Alltrumplaststint = NYdetained_Trump.loc[NYdetained_Trump.groupby('unique_identifier')['book_in_date_time'].idxmax()]
    


Alltrumplaststint[Alltrumplaststint['state'].isin(['LA', 'TX', 'MS'])]['VD?'].value_counts()

VD?
False    4255
True     1438
Name: count, dtype: int64

In [492]:
Alltrumplaststint[Alltrumplaststint['state'].isin(['LA', 'TX', 'MS'])].info()

<class 'pandas.DataFrame'>
Index: 5693 entries, 7911 to 2616948
Data columns (total 58 columns):
 #   Column                           Non-Null Count  Dtype              
---  ------                           --------------  -----              
 0   stay_book_in_date_time           5693 non-null   datetime64[us, UTC]
 1   book_in_date_time                5693 non-null   datetime64[us, UTC]
 2   detention_facility               5693 non-null   str                
 3   book_out_date_time               5071 non-null   datetime64[us, UTC]
 4   stay_book_out_date_time          5071 non-null   datetime64[us, UTC]
 5   detention_release_reason         5071 non-null   str                
 6   stay_book_out_date               5071 non-null   str                
 7   stay_release_reason              5071 non-null   str                
 8   religion                         141 non-null    str                
 9   gender                           5693 non-null   str                
 10  marital_st

In [491]:

Alltrumplaststint[Alltrumplaststint['state'].isin(['LA', 'TX', 'MS'])]['stay_release_reason'].value_counts(dropna=False)

stay_release_reason
Removed                                                          4804
NaN                                                               622
Bonded Out - IJ                                                   137
Voluntary departure                                                40
Relief Granted by IJ                                               26
Order of recognizance                                              22
Paroled                                                            16
Order of supervision                                                8
Proceedings Terminated                                              7
Order of Recognizance - Humanitarian                                3
Transferred                                                         3
Bonded Out - Field Office                                           2
Voluntary Return                                                    2
U.S. Marshals or other agency (explain in Detention Comments)       1


In [490]:
Alltrumplaststint[Alltrumplaststint['circuit'].isin(['5th'])]['unique_identifier'].nunique()

5693

In [477]:
5693/9222

0.6173281283886358

In [479]:


Alltrumplaststint[Alltrumplaststint['state'].isin(['LA', 'TX', 'MS'])]['stay_release_reason'].value_counts(dropna=False)



stay_release_reason
Removed                                                          4804
NaN                                                               622
Bonded Out - IJ                                                   137
Voluntary departure                                                40
Relief Granted by IJ                                               26
Order of recognizance                                              22
Paroled                                                            16
Order of supervision                                                8
Proceedings Terminated                                              7
Order of Recognizance - Humanitarian                                3
Transferred                                                         3
Bonded Out - Field Office                                           2
Voluntary Return                                                    2
U.S. Marshals or other agency (explain in Detention Comments)       1


In [483]:
Alltrumplaststint.groupby('circuit')['unique_identifier'].nunique().sort_values()

circuit
8th        2
1st        6
4th       54
7th       54
6th       65
11th     114
10th     177
3rd      673
9th      715
2nd     1628
5th     5693
Name: unique_identifier, dtype: int64

In [486]:
Alltrumplaststint.query("circuit == '5th' and stay_release_reason == 'Removed'")['VD?'].value_counts()

VD?
False    3415
True     1389
Name: count, dtype: int64